In [6]:
from huggingface_hub import HfApi

hub_api = HfApi()
repo = "edipark/PuttingCupintotheDishV2"
tag_name = "v2.1"

# 1. 기존 태그 삭제 (먼저 수행)
try:
    hub_api.delete_tag(repo_id=repo, tag=tag_name, repo_type="dataset")
    print(f"기존 태그 '{tag_name}' 삭제 완료.")
except Exception:
    print("삭제할 태그가 없습니다.")

# 2. 새로운 상태로 태그 생성
hub_api.create_tag(repo_id=repo, tag=tag_name, repo_type="dataset")
print(f"새로운 태그 '{tag_name}' 생성 완료!")

기존 태그 'v2.1' 삭제 완료.
새로운 태그 'v2.1' 생성 완료!


In [15]:
from openpi.training import config as _config
from openpi.training import data_loader as _data_loader
import numpy as np

config = _config.get_config("pi05_rby1")
data_config = config.data.create(config.assets_dirs, config.model)

ds = _data_loader.create_torch_dataset(data_config, config.model.action_horizon, config.model)

x = ds[100]  # raw sample
print("keys:", x.keys())
print("state:", np.asarray(x["state"]).shape, np.asarray(x["state"]).dtype)
if "actions" in x:
    a = np.asarray(x["actions"])
    print("actions:", a.shape, a.dtype)
for k in ["head_image", "left_wrist_image", "right_wrist_image"]:
    v = x[k]
    v = np.asarray(v)
    print(k, v.shape, v.dtype)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

meta/tasks.parquet:   0%|          | 0.00/2.18k [00:00<?, ?B/s]

tasks.jsonl:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

info.json: 0.00B [00:00, ?B/s]

episodes.jsonl: 0.00B [00:00, ?B/s]

...

Fetching 68 files:   0%|          | 0/68 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/62 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

keys: dict_keys(['observation.state', 'action', 'is_first', 'is_last', 'is_terminal', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index', 'actions', 'state', 'prompt', 'head_image', 'left_wrist_image', 'right_wrist_image', 'actions_is_pad', 'task'])
state: (16,) float32
actions: (50, 16) float32
head_image (3, 224, 224) int64
left_wrist_image (3, 224, 224) int64
right_wrist_image (3, 224, 224) int64


In [11]:
import inspect
import lerobot.common.datasets.lerobot_dataset as lerobot_dataset
print("lerobot_dataset:", lerobot_dataset.__file__)
print("LeRobotDataset.__getitem__ source:")
print(inspect.getsource(lerobot_dataset.LeRobotDataset.__getitem__))

lerobot_dataset: /home/hyunjin/rby1_ws/openpi/.venv/lib/python3.11/site-packages/lerobot/common/datasets/lerobot_dataset.py
LeRobotDataset.__getitem__ source:
    def __getitem__(self, idx) -> dict:
        item = self.hf_dataset[idx]
        ep_idx = item["episode_index"].item()

        query_indices = None
        if self.delta_indices is not None:
            query_indices, padding = self._get_query_indices(idx, ep_idx)
            query_result = self._query_hf_dataset(query_indices)
            item = {**item, **padding}
            for key, val in query_result.items():
                item[key] = val

        if len(self.meta.video_keys) > 0:
            current_ts = item["timestamp"].item()
            query_timestamps = self._get_query_timestamps(current_ts, query_indices)
            video_frames = self._query_videos(query_timestamps, ep_idx)
            item = {**video_frames, **item}

        if self.image_transforms is not None:
            image_keys = self.meta.camera_key

In [14]:
import inspect
import lerobot.common.datasets.lerobot_dataset as lerobot_dataset
print(inspect.getsource(lerobot_dataset.LeRobotDataset.__init__))

    def __init__(
        self,
        repo_id: str,
        root: str | Path | None = None,
        episodes: list[int] | None = None,
        image_transforms: Callable | None = None,
        delta_timestamps: dict[list[float]] | None = None,
        tolerance_s: float = 1e-4,
        revision: str | None = None,
        force_cache_sync: bool = False,
        download_videos: bool = True,
        video_backend: str | None = None,
    ):
        """
        2 modes are available for instantiating this class, depending on 2 different use cases:

        1. Your dataset already exists:
            - On your local disk in the 'root' folder. This is typically the case when you recorded your
              dataset locally and you may or may not have pushed it to the hub yet. Instantiating this class
              with 'root' will load your dataset directly from disk. This can happen while you're offline (no
              internet connection).

            - On the Hugging Face Hub at the 

In [12]:
from lerobot.common.datasets import lerobot_dataset
meta = lerobot_dataset.LeRobotDatasetMetadata(repo)
print("meta.state_dim:", getattr(meta, "state_dim", None))
print("meta.action_dim:", getattr(meta, "action_dim", None))
print("meta.state_keys:", getattr(meta, "state_keys", None))
print("meta.action_keys:", getattr(meta, "action_keys", None))

meta.state_dim: None
meta.action_dim: None
meta.state_keys: None
meta.action_keys: None


In [13]:
from lerobot.common.datasets import lerobot_dataset
meta = lerobot_dataset.LeRobotDatasetMetadata(repo)
print("meta.root:", meta.root)
print("meta.features.state:", meta.info["features"].get("state"))
print("meta.features.actions:", meta.info["features"].get("actions"))
print("meta.features.observation.state:", meta.info["features"].get("observation.state"))
print("meta.features.action:", meta.info["features"].get("action"))

meta.root: /home/hyunjin/.cache/huggingface/lerobot/edipark/PuttingCupintotheDishV2
meta.features.state: {'dtype': 'float32', 'shape': (26,), 'names': None}
meta.features.actions: {'dtype': 'float32', 'shape': (26,), 'names': None}
meta.features.observation.state: {'dtype': 'float32', 'shape': (26,), 'names': None}
meta.features.action: {'dtype': 'float32', 'shape': (26,), 'names': None}


In [10]:
from datasets import load_dataset
import numpy as np

ds = load_dataset(repo, data_files="data/chunk-000/*.parquet", split="train")
print(ds.features)

x = ds[0]
for k in ["state", "actions", "observation.state", "action"]:
    if k in x:
        v = np.asarray(x[k])
        print(k, v.shape, v.dtype)

Resolving data files:   0%|          | 0/62 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/62 [00:00<?, ?it/s]

{'observation.state': Sequence(feature=Value(dtype='float32', id=None), length=16, id=None), 'action': Sequence(feature=Value(dtype='float32', id=None), length=16, id=None), 'is_first': Value(dtype='bool', id=None), 'is_last': Value(dtype='bool', id=None), 'is_terminal': Value(dtype='bool', id=None), 'timestamp': Value(dtype='float32', id=None), 'frame_index': Value(dtype='int64', id=None), 'episode_index': Value(dtype='int64', id=None), 'index': Value(dtype='int64', id=None), 'task_index': Value(dtype='int64', id=None), 'actions': Sequence(feature=Value(dtype='float32', id=None), length=16, id=None), 'state': Sequence(feature=Value(dtype='float32', id=None), length=16, id=None), 'prompt': Value(dtype='string', id=None), 'head_image': Sequence(feature=Sequence(feature=Sequence(feature=Value(dtype='uint8', id=None), length=224, id=None), length=224, id=None), length=3, id=None), 'left_wrist_image': Sequence(feature=Sequence(feature=Sequence(feature=Value(dtype='uint8', id=None), length=

In [ ]:
ds.set_format(type="numpy")
x = ds[0]
print(x.keys())
print(x["head_image"].dtype)
print(x["left_wrist_image"].dtype)
print(x["right_wrist_image"].dtype)


dict_keys(['observation.state', 'action', 'is_first', 'is_last', 'is_terminal', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index', 'actions', 'state', 'prompt', 'head_image', 'left_wrist_image', 'right_wrist_image'])
int64
int64
int64
